In [ ]:
import matplotlib
# matplotlib.use('Qt6')  # 使用 Qt5 后端
import matplotlib.pyplot as plt
from matplotlib import font_manager
from concurrent.futures import ThreadPoolExecutor
from functools import partial
import matplotlib.image as mpimg
import numpy as np
import pylab
pylab.rcParams['figure.figsize'] = (15.0, 8.0)

from einops import rearrange, repeat

from compute import *

# hype parameter
model_name = 'EdSr'
control_name = 'MD'
benchmark_name = 'BM'
""" Temp Press PotEng KinEng Enthalpy E_vdwl E_coul E_pair E_bond E_angle E_dihed E_long E_tail E_mol Ecouple Econserve TotEng Lx Ly Lz"""
energy_unit = "kcal $\\cdot$ mol$^{-1}$"
press_unit = "ATM"
temperature = "K"
distance_unit = "Angstrom"
time_unit = "fs"
time_scale = {
    "fs": 1,
    "ps": 1000,
    "ns": 1000000,
}
xaxis_time_unit = 'ps'
thermo_style_unit = {
    'temp'     : f"Kelvin ({temperature})",    'Temp'     : f'Kelvin ({temperature})',
    'press'    : f'ATMosphere ({press_unit})', 'Press'    : f'ATMosphere ({press_unit})',
    "pe"       : f'energy ({energy_unit})',    'PotEng'   : f'energy ({energy_unit})',
    "ke"       : f'energy ({energy_unit})',    'KinEng'   : f'energy ({energy_unit})',
    "enthalpy" : f'energy ({energy_unit})',    'Enthalpy' : f'energy ({energy_unit})',
    "evdwl"    : f'energy ({energy_unit})',    'E_vdwl'   : f'energy ({energy_unit})',
    "ecoul"    : f'energy ({energy_unit})',    'E_coul'   : f'energy ({energy_unit})',
    "epair"    : f'energy ({energy_unit})',    'E_pair'   : f'energy ({energy_unit})',
    "ebond"    : f'energy ({energy_unit})',    'E_bond'   : f'energy ({energy_unit})',
    "eangle"   : f'energy ({energy_unit})',    'E_angle'  : f'energy ({energy_unit})',
    "edihed"   : f'energy ({energy_unit})',    'E_dihed'  : f'energy ({energy_unit})',
    "eimp"     : f'energy ({energy_unit})',
    "elong"    : f'energy ({energy_unit})',    'E_long'   : f'energy ({energy_unit})',
    "etail"    : f'energy ({energy_unit})',    'E_tail'   : f'energy ({energy_unit})',
    "emol"     : f'energy ({energy_unit})',    'E_mol'    : f'energy ({energy_unit})',
    "ecouple"  : f'energy ({energy_unit})',    'Ecouple'  : f'energy ({energy_unit})',
    "econserve": f'energy ({energy_unit})',    'Econserve': f'energy ({energy_unit})',
    "etotal"   : f'energy ({energy_unit})',    'TotEng'   : f'energy ({energy_unit})',
    'lx'       : f'length ({distance_unit})',  'Lx'       : f'length ({distance_unit})',
    'ly'       : f'length ({distance_unit})',  'Ly'       : f'length ({distance_unit})',
    'lz'       : f'length ({distance_unit})',  'Lz'       : f'length ({distance_unit})',
}

title_mapping = {
     'temp'     : r'Temperature',                                                'Temp'     : r'Temperature',
     'press'    : r'Pressure',                                                   'Press'    : r'Pressure',
     "pe"       : r'Potential energy',                                           'PotEng'   : r'potential energy',
     "ke"       : r'Kinetic energy',                                             'KinEng'   : r'Kinetic energy',
     "enthalpy" : r'Total energy (pe + ke)',                                     'Enthalpy' : r'Total energy (pe + ke)',
     "evdwl"    : r'Van der Waals pairwise energy',                              'E_vdwl'   : r'Van der Waals pairwise energy',
     "ecoul"    : r'Coulombic pairwise energy',                                  'E_coul'   : r'Coulombic pairwise energy',
     "epair"    : r'Pairwise energy',                                            'E_pair'   : r'Pairwise energy',
     "ebond"    : r'Bond energy',                                                'E_bond'   : r'Bond energy',
     "eangle"   : r'Angle energy',                                               'E_angle'  : r'Angle energy',
     "edihed"   : r'Dihedral energy',                                            'E_dihed'  : r'Dihedral energy',
     "eimp"     : r'Improper energy',
     "elong"    : r'Long-range kspace energy',                                   'E_long'   : r'Long-range kspace energy',
     "etail"    : r'Van der Waals energy long-range tail correction',            'E_tail'   : r'Van der Waals energy long-range tail correction',
     "emol"     : r'Intramolecular energy',                                      'E_mol'    : r'Intramolecular energy',
     "ecouple"  : r'Cumulative energy change due to thermo/baro statting fixes', 'Ecouple'  : r'Cumulative energy change due to thermo/baro statting fixes',
     "econserve": r'Etotal + ecouple',                                           'Econserve': r'Etotal + ecouple',
     "etotal"   : r'Total energy',                                               'TotEng'   : r'Total energy',
     'lx'       : r'Length of x-axis',                                           'Lx'       : r'Length of x-axis',
     'ly'       : r'Length of y-axis',                                           'Ly'       : r'Length of y-axis',
     'lz'       : r'Length of z-axis',                                           'Lz'       : r'Length of z-axis',
     'Rg'       : r'Radius of gyration',                                         'RG'       : r'Radius of gyration',
     'rmsd'     : r'RMSD',                                                       'RMSD'     : r'RMSD',
     'msd'      : r'MSD',                                                        'MSD'      : r'MSD',
}


np.set_printoptions(threshold = np.inf)

fontsize = font_manager.FontProperties(size = 9)
tick_fontsize = font_manager.FontProperties(size = 8)
title_fontsize = font_manager.FontProperties(size = 10)
legned_fontsize = font_manager.FontProperties(size = 5.5)
item_fontsize = font_manager.FontProperties(size = 12)

tight_layout_arg = dict(
    top=0.937,
    bottom=0.105,
    left=0.048,
    right=0.991,
    hspace=0.212,
    wspace=0.147
)

row, col = 2, 2

fig = plt.figure(figsize=[10*col,20*row], dpi=256)

pic_idx = 0
subgraph_item = 0 + ord('a') - 1
item_pos = (-0.05, 1.15)
title_pos = (0.5, 1.12)

# fig.subplots_adjust(**tight_layout_arg)

In [ ]:
maxIter   : int   = 100
# intv      : int   = 100
basis     : float = 0.01
start     : int   = 1
end       : int   = 10000
group     : str   = None
prefix    : str   = 'Indole'

edsr_params = [
    f'frames{start}_{end}.npz', 
    f'GlobalVariable.npz',
]

edsr_path = [
    f'data/{model_name}_nve_basis{basis}_intv{intv}_iter{maxIter}/' if prefix == '' else f'data/{prefix}_{model_name}_nve_basis{basis}_intv{intv}_iter{maxIter}/'
    for intv, maxIter  in zip([25, 50, 100, 200, 400], [100, 100, 100, 50, 100])
]

benchmark_path = [
    f'data/benchmark_nve_basis{basis}_intv{intv}/' if prefix == '' else f'data/{prefix}_benchmark_nve_basis{basis}_intv{intv}/'
    for intv in [25, 50, 100, 200, 400]
]

# mass shape: (natoms,), id shape: (natoms,),  x shape: (ntrajs, natoms, 3), v shape: (ntrajs, natoms, 3)
with ThreadPoolExecutor(max_workers = 10) as executor:

    exefunc = partial(extraction, data_file = edsr_params[0], global_file = edsr_params[1])

    edsr_futures = list(executor.map(exefunc, edsr_path))

    benchmark_futures = list(executor.map(exefunc, benchmark_path))



In [ ]:
def process_coordinate(benchmark_x, benchmark_boundary, edsr_x, edsr_boundary, times, intv):

    # coordinate figure

    xdiff_bt = traj_abs_diff(benchmark_x, benchmark_boundary, edsr_x, edsr_boundary)

    frame_xdiff_bt = np.mean(xdiff_bt, axis = (-1,  -2))

    skip = 1
    if skip > 1:
        gmean_xdiff_bt = np.pad(np.mean(frame_xdiff_bt.reshape(-1, skip), axis = -1), pad_width = (1, 0), mode = 'constant', constant_values = frame_xdiff_bt[0])
        group_times = np.pad(times[::skip] + (skip - 1)*intv*basis, pad_width = (1, 0), mode = 'constant', constant_values = 0)
    else:
        gmean_xdiff_bt = frame_xdiff_bt
        group_times = times

    ax = plt.subplot(row, col, 1)
    plt.text(*item_pos, 'a', transform = ax.transAxes, fontsize = item_fontsize.get_size(), fontweight = 'bold', va = 'top', ha = 'left')

    # ! title of column 
    plt.text(*title_pos, "Coordinate MAE ($\mathrm{\AA}$)", transform = ax.transAxes, fontsize = title_fontsize.get_size(), fontweight = 'bold', va = 'center', ha = 'center')
    plt.xticks([])

    plt.plot(group_times, gmean_xdiff_bt, label = f"{intv * basis} fs", linewidth = 2)

    plt.ticklabel_format(axis = 'x', style = 'plain', scilimits = (0, 2), useMathText = True, useLocale = False)

    plt.yscale('log')
    plt.ylim(bottom = 1e-6, top = 0.2)
    plt.tick_params(axis = 'both', labelsize = tick_fontsize.get_size())

    plt.legend(fontsize = legned_fontsize.get_size(), loc = 'lower right', ncol = 2)
    plt.subplots_adjust(**tight_layout_arg)

In [ ]:
def process_rmsd(benchmark_x, benchmark_mass, benchmark_boundary, edsr_x, edsr_mass, edsr_boundary, times, intv, use_bm = False):
    # RMSD
    ppty = 'rmsd'
    
    if use_bm: 
        benchmark_RMSD = compute_RMSD(benchmark_x, benchmark_mass, 0, benchmark_boundary)
    edsr_RMSD = compute_RMSD(edsr_x, edsr_mass, 0, edsr_boundary)

    skip = 1
    if skip > 1:
        group_times = np.pad(times.reshape(-1, skip)[:, -1], pad_width = (1, 0), mode = 'constant', constant_values = 0)
        skip_brmsd = np.pad(np.mean(benchmark_RMSD.reshape(-1, skip), axis = -1), pad_width = (1, 0), mode = 'constant', constant_values = benchmark_RMSD[0])
        skip_trmsd = np.pad(np.mean(edsr_RMSD.reshape(-1, skip), axis = -1), pad_width = (1, 0), mode = 'constant', constant_values = edsr_RMSD[0])
    else:
        group_times = times


    ax = plt.subplot(row, col, 2)
    plt.text(*item_pos, 'b', transform = ax.transAxes, fontsize = item_fontsize.get_size(), fontweight = 'bold', va = 'top', ha = 'left')

    # attn title of column 
    plt.text(*title_pos, f"{title_mapping[ppty]} ($\\mathrm{{\\AA}}$)", fontsize = title_fontsize.get_size(), transform = ax.transAxes, fontweight = 'bold', va = 'center', ha = 'center')
    if use_bm:
        plt.plot(times, benchmark_RMSD, label = f"{benchmark_name} ({basis} fs)", linewidth = 2)
    plt.plot(times, edsr_RMSD, label = f"{intv* basis} fs", linewidth = 1, alpha = 0.7)

    plt.tick_params(axis = 'both', labelsize = tick_fontsize.get_size())
    plt.yscale('log')

    plt.legend(fontsize = legned_fontsize.get_size(), loc = 'lower right', ncol = 2)
    plt.xticks([])
    plt.subplots_adjust(**tight_layout_arg)


In [ ]:
def process_potential(benchmark_ppties, benchmark_heads, edsr_ppties, edsr_heads, times, intv, use_bm):
    # potential energy
    ppty = 'pe'

    tot_bidx, = np.where(benchmark_heads == ppty); btot = benchmark_ppties[:, tot_bidx]
    tot_eidx, = np.where(edsr_heads == ppty); etot = edsr_ppties[:, tot_eidx]
    print(tot_bidx, tot_eidx)

    skip = 400 // intv * 5
    if skip > 1:
        group_times = np.pad(times.reshape(-1, skip)[:, -1], pad_width = (1, 0), mode = 'constant', constant_values = 0)
        skip_btot = np.pad(np.mean(btot.reshape(-1, skip), axis = -1), pad_width = (1, 0), mode = 'constant', constant_values = btot[0])
        skip_etot = np.pad(np.mean(etot.reshape(-1, skip), axis = -1), pad_width = (1, 0), mode = 'constant', constant_values = etot[0])
    else:
        group_times = times
        skip_btot = btot
        skip_etot = etot

    print(intv, group_times.shape, skip_btot.shape, skip_etot.shape)

    ax = plt.subplot(row, col, 3)
    plt.text(*item_pos, 'c', transform = ax.transAxes, fontsize = item_fontsize.get_size(), fontweight = 'bold', va = 'top', ha = 'left')

    # attn title of column 
    plt.text(*title_pos, f"{title_mapping[ppty]} ({energy_unit})", fontsize = title_fontsize.get_size(), transform = ax.transAxes, fontweight = 'bold', va = 'center', ha = 'center')
    if use_bm:
        plt.plot(group_times, skip_btot, label = f"{benchmark_name} ({basis} fs)", linewidth = 2)
    plt.plot(group_times, skip_etot, label = f"{intv* basis} fs", linewidth = 1, alpha = 0.5)

    plt.xlabel(f'time (fs)', fontproperties = fontsize, fontweight = 'bold')

    plt.tick_params(axis = 'both', labelsize = tick_fontsize.get_size())
    plt.legend(fontsize = legned_fontsize.get_size(), loc = 'lower left', ncol = 2)
    plt.subplots_adjust(**tight_layout_arg)

In [ ]:
def process_totalenergy(benchmark_ppties, benchmark_heads, edsr_ppties, edsr_heads, times, intv, use_bm):
    # total energy 
    ppty = 'etotal'

    tot_bidx, = np.where(benchmark_heads == ppty); btot = benchmark_ppties[:, tot_bidx]
    tot_eidx, = np.where(edsr_heads == ppty); etot = edsr_ppties[:, tot_eidx]
    print(tot_bidx, tot_eidx)

    skip = 400 // intv * 5
    if skip > 1:
        group_times = np.pad(times.reshape(-1, skip)[:, -1], pad_width = (1, 0), mode = 'constant', constant_values = 0)
        skip_btot = np.pad(np.mean(btot.reshape(-1, skip), axis = -1), pad_width = (1, 0), mode = 'constant', constant_values = btot[0])
        skip_etot = np.pad(np.mean(etot.reshape(-1, skip), axis = -1), pad_width = (1, 0), mode = 'constant', constant_values = etot[0])
    else:
        group_times = times
        skip_btot = btot
        skip_etot = etot


    ax = plt.subplot(row, col, 4)
    plt.text(*item_pos, 'd', transform = ax.transAxes, fontsize = item_fontsize.get_size(), fontweight = 'bold', va = 'top', ha = 'left')

    # attn title of column 
    plt.text(*title_pos, f"{title_mapping[ppty]} ({energy_unit})", fontsize = title_fontsize.get_size(), transform = ax.transAxes, fontweight = 'bold', va = 'center', ha = 'center')
    if use_bm:
        plt.plot(group_times, skip_btot, label = f"{benchmark_name} ({basis} fs)", linewidth = 2)
    plt.plot(group_times, skip_etot, label = f"{intv* basis} fs", linewidth = 1, alpha = 0.5)

    plt.xlabel(f'time (fs)', fontproperties = fontsize, fontweight = 'bold')

    plt.tick_params(axis = 'both', labelsize = tick_fontsize.get_size())
    plt.legend(fontsize = legned_fontsize.get_size(), loc = 'lower left', ncol = 2)
    plt.subplots_adjust(**tight_layout_arg)

In [ ]:
for edsr_results, benchmark_results, intv in zip(edsr_futures, benchmark_futures, [25, 50, 100, 200, 400]):

    benchmark_x, benchmark_v, benchmark_delta_t, benchmark_mass, benchmark_atype, benchmark_id, benchmark_boundary, benchmark_heads, benchmark_ppties, benchmark_init_state, benchmark_last_state = benchmark_results
    edsr_x, edsr_v, edsr_delta_t, edsr_mass, edsr_atype, edsr_id, edsr_boundary, edsr_heads, edsr_ppties, edsr_init_state, edsr_last_state = edsr_results

    times = np.arange(end)*intv*basis
    mask = times < 2500

    benchmark_x, edsr_x, times = benchmark_x[mask], edsr_x[mask], times[mask]
    benchmark_ppties, edsr_ppties = benchmark_ppties[mask], edsr_ppties[mask]
    
    process_coordinate(benchmark_x, benchmark_boundary, edsr_x, edsr_boundary, times, intv)
    process_rmsd(benchmark_x, benchmark_mass, benchmark_boundary, edsr_x, edsr_mass, edsr_boundary, times, intv, True if intv == 25 else False)
    process_potential(benchmark_ppties, benchmark_heads, edsr_ppties, edsr_heads, times, intv, True if intv == 25 else False)
    process_totalenergy(benchmark_ppties, benchmark_heads, edsr_ppties, edsr_heads, times, intv, True if intv == 25 else False)

In [ ]:
plt.subplots_adjust(**tight_layout_arg)
plt.show()